In [1]:
import pandas as pd
from textblob import TextBlob
import nltk

In [2]:
import os

file_path = "../data/raw_reviews.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError("raw_reviews.csv not found. Run 01_data_loading.ipynb first.")

df = pd.read_csv(file_path)

print("Dataset loaded for feature extraction:", len(df))

df.head()

Dataset loaded for feature extraction: 5000


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,AO94DHGC771SJ,0528881469,amazdnu,"[0, 0]",We got this GPS for my husband who is an (OTR)...,5.0,Gotta have GPS!,1370131200,"06 2, 2013"
1,AMO214LNFCEI4,0528881469,Amazon Customer,"[12, 15]","I'm a professional OTR truck driver, and I bou...",1.0,Very Disappointed,1290643200,"11 25, 2010"
2,A3N7T0DY83Y4IG,0528881469,C. A. Freeman,"[43, 45]","Well, what can I say. I've had this unit in m...",3.0,1st impression,1283990400,"09 9, 2010"
3,A1H8PY3QHMQQA0,0528881469,"Dave M. Shaw ""mack dave""","[9, 10]","Not going to write a long review, even thought...",2.0,"Great grafics, POOR GPS",1290556800,"11 24, 2010"
4,A24EV6RXELQZ63,0528881469,Wayne Smith,"[0, 0]",I've had mine for a year and here's what we go...,1.0,"Major issues, only excuses for support",1317254400,"09 29, 2011"


In [3]:
def get_sentiment(text):
    
    text = str(text)
    
    if text.strip() == "":
        return 0
    
    return TextBlob(text).sentiment.polarity


df["sentiment"] = df["reviewText"].apply(get_sentiment)

df[["reviewText","sentiment"]].head()

,reviewText,sentiment
0,We got this GPS for my husband who is an (OTR)...,0.265385
1,"I'm a professional OTR truck driver, and I bou...",0.062441
2,"Well, what can I say. I've had this unit in m...",0.099464
3,"Not going to write a long review, even thought...",0.059561
4,I've had mine for a year and here's what we go...,-0.002932


In [4]:
# Replace missing reviews with empty string
df["reviewText"] = df["reviewText"].fillna("")

# Compute review length (word count)
df["review_length"] = df["reviewText"].apply(lambda x: len(str(x).split()))

df[["reviewText", "review_length"]].head()

,reviewText,review_length
0,We got this GPS for my husband who is an (OTR)...,149
1,"I'm a professional OTR truck driver, and I bou...",427
2,"Well, what can I say. I've had this unit in m...",846
3,"Not going to write a long review, even thought...",449
4,I've had mine for a year and here's what we go...,202


In [5]:
import ast

def extract_helpful_votes(x):
    
    try:
        # convert string "[12, 15]" to list
        x = ast.literal_eval(x)
        
        if isinstance(x, list) and len(x) == 2:
            return x[0], x[1]
            
    except:
        pass
    
    return 0, 0


df[["helpful_votes","total_votes"]] = df["helpful"].apply(
    lambda x: pd.Series(extract_helpful_votes(x))
)

df[["helpful","helpful_votes","total_votes"]].head()


,helpful,helpful_votes,total_votes
0,"[0, 0]",0,0
1,"[12, 15]",12,15
2,"[43, 45]",43,45
3,"[9, 10]",9,10
4,"[0, 0]",0,0


In [ ]:
df["helpfulness_ratio"] = df["helpful_votes"] / (df["total_votes"] + 1)


df["helpfulness_ratio"] = df["helpfulness_ratio"].clip(0, 1)

df[["helpful_votes","total_votes","helpfulness_ratio"]].head()

,helpful_votes,total_votes,helpfulness_ratio
0,0,0,0.000000
1,12,15,0.750000
2,43,45,0.934783
3,9,10,0.818182
4,0,0,0.000000


In [7]:
# convert unix timestamp to datetime
df["review_time"] = pd.to_datetime(df["unixReviewTime"], unit="s")

# latest review date
latest_time = df["review_time"].max()

# calculate age of review
df["days_since_review"] = (latest_time - df["review_time"]).dt.days

df[["review_time","days_since_review"]].head()

,review_time,days_since_review
0,2013-06-02,413
1,2010-11-25,1333
2,2010-09-09,1410
3,2010-11-24,1334
4,2011-09-29,1025


In [8]:
df.to_csv("../data/processed_reviews.csv", index=False)

print("Processed dataset saved:", len(df))

Processed dataset saved: 5000
